In [1]:
import sys, re, json, base64, hashlib
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.common.config import RAW, PROCESSED, ask
from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph
from docx.oxml.ns import qn
import pandas as pd

projects = sorted(p for p in RAW.glob("*/*") if p.is_dir())
for p in projects:
    print(p.parent.name.ljust(11), p.name)

test        P08_Warranty_Returns
test        P09_Vendor_Compliance
train       P00_Pricing_Tool_UserSample
train       P01_Invoice_Matching
train       P02_Quote_Assistant
train       P03_Contract_Intelligence
train       P04_Product_Data_Enrichment
train       P05_Freight_Claims
validation  P06_Rebate_Validation
validation  P07_Email_Order_Entry


In [2]:
def iter_blocks(doc):
    for child in doc.element.body.iterchildren():
        if child.tag == qn("w:p"):
            yield Paragraph(child, doc)
        elif child.tag == qn("w:tbl"):
            yield Table(child, doc)

def table_to_md(tbl):
    rows = []
    for r in tbl.rows:
        cells = [c.text.strip().replace("\n", " ").replace("|", "/") for c in r.cells]
        if not rows or cells != rows[-1]:
            rows.append(cells)
    if not rows:
        return ""
    out = ["| " + " | ".join(rows[0]) + " |", "|" + "---|" * len(rows[0])]
    out += ["| " + " | ".join(r) + " |" for r in rows[1:]]
    return "\n".join(out)

def body_font_size(doc):
    sz = doc.styles["Normal"].font.size
    return sz.pt if sz else 11.0

In [2]:
NUM_RE = re.compile(r"^\s*(?:(?P<sec>section\s+\d+\s*[–—:-]\s*)|(?P<dec>\d+(?:\.\d+)*)[.)]?\s+|(?P<rom>[IVXLC]+)\.\s+|(?P<let>[a-z])\)\s+)", re.I)
BULLET_RE = re.compile(r"^\s*(?:[•▪◦●○■□➢►\-–—*]|step\s+\d+\s*[:.)])\s*", re.I)

def strip_number(text):
    return NUM_RE.sub("", text).strip()

def number_level(text):
    m = NUM_RE.match(text)
    if not m:
        return None
    if m.group("dec"):
        return m.group("dec").count(".") + 1
    if m.group("let"):
        return 2
    return 1

def run_props(para):
    runs = [r for r in para.runs if r.text.strip()]
    if not runs:
        return False, False, None
    bold = all(r.bold or (r.style is not None and r.style.font.bold) for r in runs)
    under = all(bool(r.underline) for r in runs)
    sizes = [r.font.size.pt for r in runs if r.font.size]
    return bold, under, (max(sizes) if sizes else None)

def is_indented(para):
    ind = para.paragraph_format.left_indent
    return ind is not None and ind.pt >= 9

def classify(para, base_size):
    """Return (kind, level, strong). kind: heading / list / para / None (empty).
    strong=True means the heading comes from a Word Heading/Title style (trusted)."""
    text = para.text.strip()
    if not text:
        return None, None, False
    style = para.style.name if para.style is not None else ""
    if style.lower().startswith("heading"):
        return "heading", int(re.sub(r"\D", "", style) or 1), True
    if style.lower().startswith("title"):
        return "heading", 1, True
    # ---- list items first: Word list styles, bullet/dash characters, "Step 1:", indented lines
    if "list" in style.lower() or (para._p.pPr is not None and para._p.pPr.numPr is not None):
        return "list", None, False
    if BULLET_RE.match(text) and not NUM_RE.match(text):
        return "list", None, False
    bold, under, size = run_props(para)
    big = size is not None and size >= base_size + 1.5
    if is_indented(para) and not big:
        return "list", None, False

    words = len(text.split())
    short = len(text) <= 90 and words <= 12
    ends_sentence = text.endswith((".", ";", ",")) and not re.search(r"\b(e\.g|etc)\.$", text)
    letters = re.sub(r"[^A-Za-z]", "", text)
    caps = len(letters) >= 4 and letters.isupper()
    numbered = number_level(text)

    if not short or ends_sentence:
        return "para", None, False
    if bold and text.endswith(":"):
        return "heading", 3, False              # bold label like "Dependencies:"
    if text.endswith(":"):
        return "para", None, False
    if bold or under or caps or big:
        if numbered:
            return "heading", min(numbered, 3), False
        return "heading", (1 if (big or caps or under) else 2), False
    if numbered and words <= 8 and strip_number(text)[:1].isupper():
        return "heading", min(numbered, 3), False   # plain numbered heading like "4. Scope"
    return "para", None, False

def demote_runs(blocks):
    """Weak headings that sit directly next to another weak heading of the SAME level,
    with no content between them, are really a list (e.g. '1. Reduce cost' / '2. Faster quotes')."""
    idx = [i for i, b in enumerate(blocks) if b["kind"] is not None]
    for a, b in zip(idx, idx[1:]):
        x, y = blocks[a], blocks[b]
        if (x["kind"] == "heading" and y["kind"] == "heading" and not x["strong"] and not y["strong"]
                and x["level"] == y["level"] and x["level"] < 3):
            x["demote"] = y["demote"] = True
    for b in blocks:
        if b.get("demote"):
            b["kind"], b["level"] = "list", None
    return blocks

# quick check on a messy BRD
d = Document(next(RAW.glob("validation/P06*/*_BRD.docx")))
base = body_font_size(d)
for blk in list(iter_blocks(d))[:40]:
    if isinstance(blk, Paragraph):
        k, l, s = classify(blk, base)
        if k: print(f"{k:8} {l or '':2} {blk.text[:70]}")

NameError: name 'body_font_size' is not defined

In [ ]:
from docx import Document
from docx.shared import Pt

doc = Document()

# Normal paragraph
doc.add_paragraph(
    "This is a normal paragraph containing enough words to be treated as a paragraph."
)

# Numbered heading
doc.add_paragraph("1. Introduction")

# Nested numbered heading
doc.add_paragraph("1.1 Scope")

# Roman numeral heading
doc.add_paragraph("IV. Requirements")

# Letter heading
doc.add_paragraph("a) Dependencies")

# Section heading
doc.add_paragraph("Section 3 - Testing")

# Bullet item
doc.add_paragraph("• First bullet")

# Dash bullet
doc.add_paragraph("- Second bullet")

# Step bullet
doc.add_paragraph("Step 1: Collect data")

# ALL CAPS heading
doc.add_paragraph("PROJECT OVERVIEW")

# Bold heading
p = doc.add_paragraph()
r = p.add_run("Implementation")
r.bold = True

# Bold label ending with colon
p = doc.add_paragraph()
r = p.add_run("Dependencies:")
r.bold = True

# Indented text
p = doc.add_paragraph("Indented item")
p.paragraph_format.left_indent = Pt(12)

base = 11

for para in doc.paragraphs:
    kind, level, strong = classify(para, base)
    print(
        f"{kind!s:8} level={level!s:4} strong={strong} text='{para.text}'"
    )

In [4]:
TOC_RE = re.compile(r"^(table of contents|contents)$", re.I)

def norm(s):
    return re.sub(r"[^a-z0-9 ]", "", strip_number(s).lower()).strip()

def parse_docx(path):
    doc = Document(path)
    base = body_font_size(doc)

    # pass 1: classify every block
    blocks = []
    for blk in iter_blocks(doc):
        if isinstance(blk, Table):
            md = table_to_md(blk)
            blocks.append({"kind": "table" if md else None, "level": None, "strong": False, "text": md})
        else:
            k, l, s = classify(blk, base)
            blocks.append({"kind": k, "level": l, "strong": s, "text": blk.text.strip()})

    # pass 2: table of contents -> skip entries until the first entry reappears in the body
    toc, i = [], 0
    while i < len(blocks):
        b = blocks[i]
        if b["kind"] in ("heading", "para", "list") and TOC_RE.match(strip_number(b["text"]).rstrip(":")):
            b["kind"] = None; j = i + 1
            while j < len(blocks) and len(toc) < 40:
                t = blocks[j]
                if t["kind"] is None:
                    j += 1; continue
                if t["kind"] == "table" or len(t["text"]) >= 90 or (toc and norm(t["text"]) == norm(toc[0])):
                    break
                toc.append(t["text"]); t["kind"] = None; j += 1
            if j < len(blocks) and toc and norm(blocks[j]["text"]) == norm(toc[0]) and blocks[j]["kind"] != "heading":
                blocks[j]["kind"], blocks[j]["level"] = "heading", number_level(blocks[j]["text"]) or 1
            break
        i += 1

    # pass 3: list runs that looked like headings
    blocks = demote_runs(blocks)

    # pass 4: build markdown + sections
    lines, sections = [], []
    cur = {"heading": "(front matter)", "level": 0, "text": []}
    def close():
        if cur["text"] or cur["level"] > 0:
            sections.append({"heading": cur["heading"], "level": cur["level"], "text": "\n".join(cur["text"]).strip()})

    for b in blocks:
        kind, text = b["kind"], b["text"]
        if kind is None:
            continue
        if kind == "heading":
            close()
            title = strip_number(text).rstrip(":").strip() or text
            cur = {"heading": title, "level": b["level"], "text": []}
            lines.append("\n" + "#" * b["level"] + " " + title)
        elif kind == "list":
            item = "- " + strip_number(BULLET_RE.sub("", text))
            lines.append(item); cur["text"].append(item)
        else:                                   # para or table
            lines.append(text); cur["text"].append(text)
    close()
    return "\n".join(lines).strip(), sections, toc

def section_stats(secs):
    body = [s for s in secs if s["level"] > 0]
    empty = 0
    for i, s in enumerate(body):
        nxt = body[i + 1] if i + 1 < len(body) else None
        has_children = nxt is not None and nxt["level"] > s["level"]
        if not s["text"] and not has_children:
            empty += 1                           # heading with no text AND no sub-sections
    return len(body), empty

md, secs, toc = parse_docx(next(RAW.glob("validation/P06*/*_BRD.docx")))
print("TOC entries found:", len(toc), "| sections / empty:", section_stats(secs))
for s in secs:
    print(("  " * max(s["level"] - 1, 0)) + f"[H{s['level']}] {s['heading'][:60]}  ({len(s['text'].split())} words)")

TOC entries found: 0 | sections / empty: (30, 0)
[H1] Business Requirement Specification – Supplier Rebate Claim V  (13 words)
[H1] Purpose  (62 words)
[H1] Current State  (46 words)
[H1] Objectives  (35 words)
[H1] Scope of Work  (81 words)
[H1] User Roles  (51 words)
[H1] User Stories  (82 words)
[H1] Functional Requirements (by MVP)  (105 words)
[H1] Quality Attributes  (44 words)
[H1] Solution Architecture  (2 words)
[H1] Data Requirements  (132 words)
[H1] Interfaces and Dependencies on Systems  (0 words)
  [H2] Identity & Access  (3 words)
  [H2] Enterprise Data  (10 words)
  [H2] AI Services  (6 words)
  [H2] Data & Storage  (11 words)
[H1] Compliance  (10 words)
[H1] Assumptions & Constraints  (18 words)
[H1] External Dependencies  (14 words)
[H1] Timeline & Milestones  (8 words)
[H1] Acceptance Criteria  (23 words)
[H1] Risk Register  (34 words)
[H1] ROI  (6 words)
[H1] Approvals  (17 words)
[H1] Appendix  (0 words)
  [H2] Rebate Claim Validation MVP1 Scope Overview  (21 words

In [5]:
ARCH_SYSTEM = ("You convert software architecture diagrams into precise text for test planning. "
               "Describe ONLY what is visible. Use the exact component and label names from the diagram.")
ARCH_PROMPT = """Describe this solution architecture diagram using exactly these headings:
TITLE:
USERS / CLIENTS:
IDENTITY & ENTRY POINTS: (e.g. Azure AD, App Gateway)
APPLICATION SERVICES:
AI SERVICES:
DATA STORES:
INTEGRATION / MESSAGING:
EXTERNAL / ENTERPRISE SYSTEMS:
SUPPORTING SERVICES: (monitoring, CI/CD, secrets)
CONNECTIONS: one line per arrow -> source -> target | label (protocol / auth / purpose)
ENVIRONMENT NOTES: (subscription, non-production, network boundaries)"""

CACHE = PROCESSED / "_cache"; CACHE.mkdir(parents=True, exist_ok=True)

def describe_arch(img_path):
    data = img_path.read_bytes()
    key = hashlib.sha1(data).hexdigest()[:16]
    cache = CACHE / f"arch_{key}.md"
    if cache.exists():
        return cache.read_text(encoding="utf-8")
    mime = "image/png" if img_path.suffix.lower() == ".png" else "image/jpeg"
    b64 = base64.b64encode(data).decode()
    text = ask(ARCH_SYSTEM, [{"type": "input_text", "text": ARCH_PROMPT},
                             {"type": "input_image", "image_url": f"data:{mime};base64,{b64}"}], effort="low")
    cache.write_text(text, encoding="utf-8")
    return text

print(describe_arch(next(RAW.glob("validation/P06*/*_Architecture.*")))[:1500])

TITLE:
Supplier Rebate Claim Validation - Solution Architecture (Non-production)

USERS / CLIENTS:
Rebate Analysts
Finance Managers

IDENTITY & ENTRY POINTS: (e.g. Azure AD, App Gateway)
Azure AD (Entra ID)
App Gateway

APPLICATION SERVICES:
Web UI
Agreement Service
Calculation Service
Azure Function (Batch Scheduler)
APIM

AI SERVICES:
Azure Document Intelligence
Azure OpenAI

DATA STORES:
Azure Blob Storage
Azure PostgreSQL
Delta Lake EUDM/IAP (Databricks SQL)

INTEGRATION / MESSAGING:
Azure Function (Batch Scheduler)
APIM

EXTERNAL / ENTERPRISE SYSTEMS:
Delta Lake EUDM/IAP (Databricks SQL)

SUPPORTING SERVICES: (monitoring, CI/CD, secrets)
Datadog
GitHub Actions
Key Vault
Azure Monitor

CONNECTIONS: one line per arrow -> source -> target | label (protocol / auth / purpose)
Rebate Analysts -> App Gateway | HTTPS/443 AD/SSO  
Finance Managers -> App Gateway | HTTPS/443 AD/SSO  
Azure AD (Entra ID) -> App Gateway | HTTPS/443 AD/SSO / OAuth 2.0  
App Gateway -> Web UI | HTTPS/443  
Web 

In [6]:
def find(folder, pattern):
    hits = sorted(folder.glob(pattern))
    return hits[0] if hits else None

report = []
for proj in projects:
    out = PROCESSED / proj.parent.name / proj.name
    out.mkdir(parents=True, exist_ok=True)
    row = {"split": proj.parent.name, "project": proj.name}
    for kind, pattern in [("brd", "*BRD*.docx"), ("testplan", "*TestPlan*.docx")]:
        f = find(proj, pattern)
        if not f:
            row[f"{kind}_sections"] = "MISSING"; continue
        md, secs, toc = parse_docx(f)
        (out / f"{kind}.md").write_text(md, encoding="utf-8")
        (out / f"{kind}_sections.json").write_text(json.dumps(secs, indent=2, ensure_ascii=False), encoding="utf-8")
        row[f"{kind}_sections"], row[f"{kind}_empty"] = section_stats(secs)
        row[f"{kind}_words"] = len(md.split())
        row[f"{kind}_toc"] = len(toc) > 0
    img = find(proj, "*Architecture.*")
    if img:
        (out / "architecture.md").write_text(describe_arch(img), encoding="utf-8")
        row["arch"] = "ok"
    else:
        row["arch"] = "MISSING"
    report.append(row)

df = pd.DataFrame(report)
df.to_csv(PROCESSED / "parse_report.csv", index=False)
df

,split,project,brd_sections,brd_empty,brd_words,brd_toc,testplan_sections,testplan_empty,testplan_words,testplan_toc,arch
0,test,P08_Warranty_Returns,32,0,1032,True,23,0,1296,False,ok
1,test,P09_Vendor_Compliance,29,0,1012,False,29,0,1227,False,ok
2,train,P00_Pricing_Tool_UserSample,30,2,1477,True,24,0,1868,False,ok
3,train,P01_Invoice_Matching,33,0,1480,True,30,0,1671,False,ok
4,train,P02_Quote_Assistant,33,0,1401,True,24,0,1612,False,ok
5,train,P03_Contract_Intelligence,32,0,1196,False,25,0,1478,False,ok
6,train,P04_Product_Data_Enrichment,28,0,1102,True,32,0,1237,False,ok
7,train,P05_Freight_Claims,34,0,1124,True,27,0,1353,True,ok
8,validation,P06_Rebate_Validation,30,0,993,False,30,0,1248,False,ok
9,validation,P07_Email_Order_Entry,38,0,1015,True,32,0,1202,True,ok


In [7]:
def show(project_prefix, kind="testplan", n_sections=6):
    proj = next(p for p in (PROCESSED).glob("*/*") if p.name.startswith(project_prefix))
    secs = json.loads((proj / f"{kind}_sections.json").read_text(encoding="utf-8"))
    for s in secs[: n_sections]:
        print("=" * 90); print(f"[H{s['level']}] {s['heading']}"); print(s["text"][:600])

show("P02", "testplan")

[H1] Master Test Plan – Quote Assistant
Version: 1.0
Application: Quote Assistant
Prepared by: QA Lead, Quality Engineering
Reviewed by: Test Manager
Test phase: MVP1
Audience: Business Stakeholders, Product Owners, Architects, Development Team and QA Team
Purpose. This plan defines the quality approach for validating Quote Assistant MVP1, from RFQ intake through line extraction, catalog matching, draft quote review and export. Salesforce CPQ creation, pricing and learning from corrections are MVP2 and are not release blockers for MVP1.
[H2] References
Quote Assistant BRD. Quote Assistant architecture diagram.
[H1] Test Objectives
Verify extraction of customer part number, description, manufacturer, quantity and unit of measure from RFQ emails, PDF, DOCX and XLSX. Confirm product matching returns the correct catalog item as top-1 for at least 80% of lines and up to three alternatives. Validate the 70% confidence threshold, highlighting and manual review behaviour. Verify the model only